In [1]:
import pandas as pd
import re
import math
import json
import requests
from io import StringIO

TOUR_NAME = "25_merida"
POKEDATA_ID = '0000148'
CATEGORY = 'masters'
POKEDATA_CSV = f'https://pokedata.ovh/standings/{POKEDATA_ID}/{CATEGORY}/data.csv?'
DAY1_ROUNDS = 8

def remove_brackets(input_string):
    result = re.sub(r'\s*\[.*?\]\s*', '', input_string)
    return result

def normalize_name(name):
    name = name.lower()
    name = re.sub(r'[^a-z0-9_]', '_', name)
    return name



In [2]:
pairings_df = pd.read_csv(StringIO(requests.get(POKEDATA_CSV).content.decode('utf-8')), sep='\t', header=None, encoding='utf-8')
pairings_df.rename(columns={0:'Player',1:'Opponent',2:'Result',3:'Points',4:'Round'}, inplace=True)
pairings_df['Player'] = pairings_df['Player'].apply(remove_brackets).apply(normalize_name)
pairings_df['Opponent'] = pairings_df['Opponent'].apply(remove_brackets).apply(normalize_name)
pairings_df = pairings_df[(pairings_df['Opponent'] != 'BYE') & (pairings_df['Opponent'] != 'LATE')]

In [3]:
from bs4 import BeautifulSoup

def parse_limitless(html_content):
    soup = BeautifulSoup(html_content, 'html.parser')
    table = soup.find('table', class_='data-table striped')
    data_tooltips = []
    for row in table.find_all('tr')[1:]:
        deck = row.find('span')['data-tooltip']
        data_tooltips.append(deck if deck else 'Other')
    return data_tooltips

deck_df = pd.DataFrame()
for tournament in ['limitless.html']:
    with open(tournament, 'r', encoding='utf-8') as file:
        html_content = file.read()
        it_deck_df = pd.DataFrame()
        it_deck_df['Deck'] = parse_limitless(html_content)
        it_deck_df['Player'] = pairings_df['Player'].unique()[:len(it_deck_df)]
        placements = []
        for i in range(len(it_deck_df)):
            placements.append("Top {}".format(pow(2, math.ceil(math.log(i+1, 2))))) 
        it_deck_df['Placement'] = placements
    deck_df = pd.concat([deck_df, it_deck_df])


with open('archetype.json', 'r') as file:
    arch_dict = json.load(file)
    deck_df['Deck'] = deck_df['Deck'].apply(lambda x: arch_dict[x] if x in arch_dict else x)

deck_dict = deck_df.set_index('Player')['Deck'].to_dict()


In [5]:
pairings_df

,Player,Opponent,Result,Points,Round
0,azul_garcia_griego,cesar_lopez_albarran,L,0,1.0
1,azul_garcia_griego,alexis_espinoza,W,3,2.0
2,azul_garcia_griego,jorge_hern_ndez,W,6,3.0
3,azul_garcia_griego,erasmo_rodriguez,T,7,4.0
4,azul_garcia_griego,alan_gomez,W,10,5.0
...,...,...,...,...,...
9360,andres_romero_cuanca,abel_eduardo_diz_jauregui,W,7,3.0
9361,andres_romero_cuanca,manuel_alejandro_cervantes,L,7,4.0
9362,andres_romero_cuanca,christopher_saucedo,W,10,5.0
9363,andres_romero_cuanca,bladimir_mart_nez,T,11,6.0


In [4]:
deck_df

,Deck,Player,Placement
0,Regidrago,azul_garcia_griego,Top 1
1,Regidrago,mauricio_patino,Top 2
2,Raging Bolt Ogerpon,leandro_fernandes,Top 4
3,Regidrago,rolando_gabriel_coronado_jimenez,Top 4
4,Regidrago,aidan_khus,Top 8
...,...,...,...
220,Roaring Moon,marco_antonio_rodrigue_martinez,Top 256
221,Klawf Terapagos,jesus_edmundo_santillano_mier,Top 256
222,Charizard Pidgeot,adrian_canul,Top 256
223,Charizard Pidgeot,carlos_ivan_novelo_ocampo,Top 256


In [3]:
matchups_df = pd.DataFrame(columns=['Deck', 'Opposing Deck', 'Wins', 'Losses', 'Ties'])
# pairings_final_df = pd.DataFrame(columns=['Player','Opponent','Result','Points','Round'])

for index, row in pairings_df.iterrows():
    try:
        player_deck = deck_dict[row['Player']]
        opp_deck = deck_dict[row['Opponent']]
        matchup = matchups_df.loc[(matchups_df['Deck'] == player_deck) & (matchups_df['Opposing Deck'] == opp_deck)]
        # if row['Round'] == 9 and row['Points'] == 19 and row['Result'] == 'T':
        #     # print(row)
        #     continue
        if row['Result'] == 'T':
            if len(matchup) == 0:
                matchups_df.loc[len(matchups_df)] = player_deck, opp_deck, 0, 0, 1
            else:
                matchups_df.loc[matchup.index, 'Ties'] += 1
        elif row['Result'] == 'W':
            if len(matchup) == 0:
                matchups_df.loc[len(matchups_df)] = player_deck, opp_deck, 1, 0, 0
            else:
                matchups_df.loc[matchup.index, 'Wins'] += 1
        elif row['Result'] == 'L':
            if len(matchup) == 0:
                    matchups_df.loc[len(matchups_df)] = player_deck, opp_deck, 0, 1, 0
            else:
                matchups_df.loc[matchup.index, 'Losses'] += 1
        # pairings_final_df.loc[len(pairings_final_df)] = row
    except:
        continue


In [4]:
with pd.ExcelWriter(f'datasets/{TOUR_NAME}.xlsx') as writer:
    # pairings_final_df.to_excel(writer, sheet_name='pairings', index=False)
    deck_df.to_excel(writer, sheet_name='decks', index=False)
    matchups_df.to_excel(writer, sheet_name='matchups', index=False)